In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from Base_Modules.Environments import Prison
from Prison_Strategies.Basic_Strategies import *
from Base_Modules.Game_Master import Game_Master
from Base_Modules.Action import Action_History, Prison_Actions, Duel_Matrix
import pandas as pd

In [3]:
prison = Prison()
actions = prison.Get_Actions()

In [15]:
strategies_list = [
    Random_Strategy(actions=actions),
    # Random_Strategy(actions=actions, p_coop=0.1),
    # Always_Betray(actions=actions),
    # Always_Cooperate(actions=actions),
    # Patient_Unforgiving(actions=actions),
    # Copycat(actions=actions),
    Periodic(actions=actions, period=2),
]

In [5]:
def Get_Index_By_Name(strategies : dict[int, Strategy], name : str) -> int:
    for ix, (id, s) in enumerate(strategies.items()):
        if str(s) == name:
            return id
    for ix, (id, s) in enumerate(strategies.items()):
        if str(s).startswith(name):
            return id
    return -1

In [16]:
strategies = {}

for (i, s) in enumerate(strategies_list):
    strategies[i] = s
    s.Set_ID(i)

In [17]:
gm = Game_Master(prison, strategies=strategies, duel_size=2)
duel_matrix, rewards = gm.Tournament(4, Game_Master.Game_Type.All_Vs_All, True)
rewards.Sort_Total_Rewards()

In [18]:
print(duel_matrix.Get_Action_History((0, 1)).Get_Action_History())
print(Get_Index_By_Name(strategies, "Periodic"))

{0: [<Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Cooperate: 0>], 1: [<Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>]}
1


In [9]:
total_rewards = rewards.Get_Total_Rewards()
total_rewards_per_name = {str(strategies[i]):total_rewards[i] for i in total_rewards.keys()}

duel_rewards = rewards.Get_All_Duel_Rewards()
winner = next(iter(total_rewards))

## Wyniki

In [10]:
name_total_reward_df = pd.DataFrame.from_dict(total_rewards_per_name, orient="index", columns=["Total Reward"])
name_total_reward_df.index.name = "Strategy Name"
name_total_reward_df

,Total Reward
Strategy Name,
Periodic (period=1),16
Random_Strategy (p_coop=0.5),6


In [11]:
strat_names = [str(s) for s in strategies.values()]

score_matrix = pd.DataFrame(index=strat_names, columns=strat_names, dtype=object)

for (s1, s2), scores in duel_rewards.items():
    score_matrix.loc[str(strategies[s2]), str(strategies[s1])] = (scores[s2], scores[s1])
    score_matrix.loc[str(strategies[s1]), str(strategies[s2])] = (scores[s1], scores[s2])

for s in strat_names:
    score_matrix.loc[s, s] = (0, 0)

In [12]:
victory_matrix = score_matrix.apply(lambda col: col.map(lambda x: int(x[0] > x[1])))
for s in strat_names:
    victory_matrix.loc[s, s] = float("NaN")

In [13]:
def color_cell(x):
    if not isinstance(x, tuple):
        return ""
    if x[0] == 0 and x[1] == 0:
        return "background-color: black"
    elif x[0] > x[1]:
        return "background-color: green"
    elif x[0] < x[1]:
        return "background-color: darkred"
    else:
        return "background-color: gray"

styled_score_matrix = (
    score_matrix.style
    .map(color_cell)
    .set_properties(**{
        "border": "1px solid black"
    })
    .set_table_styles([
        {"selector": "th", "props": [("border", "2px solid black")]},
        {"selector": "td", "props": [("border", "2px solid black")]}
    ])
)

In [14]:
import webbrowser
store_styled_matrix_in_html = False
if store_styled_matrix_in_html:
    styled_score_matrix.to_html("styled_score_matrix.html")
    webbrowser.open_new_tab("styled_score_matrix.html")